Praktikum 2 - Pra pengolahan data terstruktur: imputasi missing value.

Strategi: Age -> mean, Cabin -> label "DECK", Embarked -> modus.
Hasil dibandingkan dengan file Titanic-Dataset-fixed.csv bawaan modul.

In [1]:
# Shim: notebook tidak punya __file__, kode ini warisan script .py.
import os
__file__ = os.path.abspath("imputasi_titanic.ipynb")
print("base:", os.path.dirname(__file__))

base: C:\code\pemmes26\js02-pemahaman-data-dan-pra-pengolahan-data\praktikum-2


In [2]:
from pathlib import Path

import numpy as np
import pandas as pd

BASE = Path(__file__).resolve().parent
DROOT = BASE.parent / "data"
OUT = BASE / "output"
OUT.mkdir(parents=True, exist_ok=True)

df = pd.read_csv(DROOT / "Titanic-Dataset.csv")
print("=== Dimensi awal ===", df.shape)

miss_before = df.isnull().sum()
print("\n=== Missing sebelum imputasi ===")
print(miss_before[miss_before > 0].to_string())

=== Dimensi awal === (891, 12)

=== Missing sebelum imputasi ===
Age         177
Cabin       687
Embarked      2


In [3]:
# ---------- Langkah 3: imputasi ----------
mean_age = df["Age"].mean()
mode_embarked = df["Embarked"].mode()[0]  # modul menulis df['Embarked'].mode (fungsi, bukan nilai)
print(f"\nNilai pengisi -> Age mean={mean_age:.4f}, "
      f"Embarked modus='{mode_embarked}', Cabin='DECK'")

df["Age"] = df["Age"].fillna(mean_age)
df["Cabin"] = df["Cabin"].fillna("DECK")
df["Embarked"] = df["Embarked"].fillna(mode_embarked)


Nilai pengisi -> Age mean=29.6991, Embarked modus='S', Cabin='DECK'


In [4]:
# ---------- Langkah 4: validasi ----------
miss_after = df.isnull().sum()
print("\n=== Missing setelah imputasi ===")
print(miss_after.to_string())
assert miss_after.sum() == 0, "masih ada missing value"

print("\n=== Tipe data setelah imputasi ===")
print(df.dtypes.to_string())
print("\nAge unik setelah imputasi:", df["Age"].nunique(),
      "| nilai mean muncul:", int((df["Age"] == mean_age).sum()), "baris")
print("Cabin 'DECK':", int((df["Cabin"] == "DECK").sum()), "baris")
print("Embarked:", df["Embarked"].value_counts().to_dict())
print("\n5 data teratas:\n", df.head().to_string())

df.to_csv(OUT / "Titanic-Dataset-imputed.csv", index=False)


=== Missing setelah imputasi ===
PassengerId    0
Survived       0
Pclass         0
Name           0
Sex            0
Age            0
SibSp          0
Parch          0
Ticket         0
Fare           0
Cabin          0
Embarked       0

=== Tipe data setelah imputasi ===
PassengerId      int64
Survived         int64
Pclass           int64
Name            object
Sex             object
Age            float64
SibSp            int64
Parch            int64
Ticket          object
Fare           float64
Cabin           object
Embarked        object

Age unik setelah imputasi: 89 | nilai mean muncul: 177 baris
Cabin 'DECK': 687 baris
Embarked: {'S': 646, 'C': 168, 'Q': 77}

5 data teratas:
    PassengerId  Survived  Pclass                                                 Name     Sex   Age  SibSp  Parch            Ticket     Fare Cabin Embarked
0            1         0       3                              Braund, Mr. Owen Harris    male  22.0      1      0         A/5 21171   7.2500  DECK    

In [5]:
# ---------- Bandingkan dengan file fixed bawaan modul ----------
fixed = pd.read_csv(DROOT / "Titanic-Dataset-fixed.csv")
print("\n=== Perbandingan dengan Titanic-Dataset-fixed.csv ===")
print("shape kita vs modul:", df.shape, fixed.shape)
for col in ["Age", "Cabin", "Embarked"]:
    same = df[col].astype(str).equals(fixed[col].astype(str))
    print(f"kolom {col}: identik={same}")
print(f"mean Age kita={df['Age'].mean():.4f} modul={fixed['Age'].mean():.4f}")
print(f"jumlah Cabin DECK kita={int((df['Cabin'] == 'DECK').sum())} "
      f"modul={int((fixed['Cabin'] == 'DECK').sum())}")


=== Perbandingan dengan Titanic-Dataset-fixed.csv ===
shape kita vs modul: (891, 12) (891, 12)
kolom Age: identik=True
kolom Cabin: identik=True
kolom Embarked: identik=False
mean Age kita=29.6991 modul=29.6991
jumlah Cabin DECK kita=687 modul=687
